# 05 — PPO LLM fine-tuning

## Learning objectives

- Verify a coherent TRL PPO API environment.
- Run local no-download reward training and four-role PPO smoke test.
- Understand optional pretrained workflow, padding and model saving.

This notebook runs a small CPU demonstration by default. Synthetic experiments are educational, not evidence of LLM alignment. Pretrained-model downloads and GPU training require explicit opt-in. All runtime outputs stay inside `RLHF_NOTEBOOKS/`.

**Source connection:** RLHF_with_PPO_Final_Code.ipynb four roles, small RM and six-episode limits; notes Chapters02/04/09; verified TRL0.22.2 PPO source/imports.

[Learning order and setup](README.md).

In [1]:
from pathlib import Path
import os, sys, json, math, random, copy, re, subprocess, hashlib
# Find this collection when launched from the repository, module, or notebook directory.
candidates = [Path.cwd(), Path.cwd()/'RLHF_NOTEBOOKS', Path.cwd()/'13_rlhf_ppo_dpo_orpo/RLHF_NOTEBOOKS']
ROOT = next((p.resolve() for p in candidates if (p/'source_inspection.json').exists()), None)
if ROOT is None:
    raise RuntimeError('Launch Jupyter from RLHF_NOTEBOOKS, its parent, or the repository root.')
RUNTIME = ROOT/'runtime'; RUNTIME.mkdir(exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(RUNTIME/'matplotlib')
os.environ['HF_HOME'] = str(RUNTIME/'huggingface')
os.environ['HF_DATASETS_CACHE'] = str(RUNTIME/'huggingface/datasets')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
random.seed(42); np.random.seed(42); torch.manual_seed(42)
torch.set_num_threads(1)
print('Python', sys.version.split()[0], 'PyTorch', torch.__version__, 'CUDA', torch.cuda.is_available())

Python 3.13.9 PyTorch 2.8.0+cpu CUDA False


## From explicit PPO to token generation
The token state is prompt plus prefix; action is next token. Policy generates, reference supplies KL anchor, reward model scores final response, critic predicts remaining shaped return. PPO retains old rollout logps separately.

$$
R_t=-\kappa[\log\pi_{old}(a_t\mid s_t)-\log\pi_{ref}(a_t\mid s_t)]+\mathbf1_{t=T-1}r_\psi(x,y).
$$
$\kappa$ KL coefficient, $r_\psi$ terminal reward, $T$ active response length. This expresses a common shaping implementation; TRL0.22.2 supports specified k1/k3 estimators and value clipping. Trainer owns token masks, GAE and old/current ratios. The CPU example has distinct PAD/EOS; pretrained recipe adds a new padding token before copying models. Reward is frozen, critic trainable. All four models consume matching vocabulary.

The TRL PPO trainer is not the old `.step(queries,responses,rewards)` value-head API. The source notebook used later experimental paths; this collection pins a release whose exact signatures were imported. Full-parameter small-model PPO is shown; this is not a four-bit memory recipe.

In [2]:
def read_preferences():
    path = ROOT.parent/'reward_preferences_dataset.jsonl'
    if path.exists():
        raw = path.read_bytes(); origin = str(path)
    else:
        # Originals are currently missing from the working tree. Read their Git blob;
        # never restore, overwrite or silently substitute invented preference records.
        gitroot = Path(subprocess.check_output(['git','rev-parse','--show-toplevel'], cwd=ROOT, text=True).strip())
        rel = (ROOT.parent/'reward_preferences_dataset.jsonl').relative_to(gitroot).as_posix()
        raw = subprocess.check_output(['git','show','HEAD:'+rel], cwd=gitroot)
        origin = 'HEAD:'+rel
    expected = json.loads((ROOT/'source_inspection.json').read_text())
    digest = next(r['sha256'] for r in expected if r['source']=='reward_preferences_dataset.jsonl')
    assert hashlib.sha256(raw).hexdigest() == digest, 'Dataset changed: re-audit before proceeding'
    rows = [json.loads(line) for line in raw.decode('utf-8').splitlines()]
    for row in rows:
        assert set(row)=={'prompt','chosen','rejected'}
        assert all(isinstance(row[k],str) and row[k].strip() for k in row)
        assert row['chosen'].strip()!=row['rejected'].strip()
    assert len({r['prompt'].strip().lower() for r in rows})==len(rows)
    print('Dataset origin:',origin,'records:',len(rows),'fields:',list(rows[0]))
    return rows
rows = read_preferences()
indices = np.random.default_rng(42).permutation(len(rows))
train_rows = [rows[i] for i in indices[:14]]
valid_rows = [rows[i] for i in indices[14:17]]
test_rows = [rows[i] for i in indices[17:]]
assert not ({r['prompt'] for r in train_rows} & {r['prompt'] for r in valid_rows+test_rows})
print('Prompt-separated split:',len(train_rows),len(valid_rows),len(test_rows))

Dataset origin: HEAD:13_rlhf_ppo_dpo_orpo/reward_preferences_dataset.jsonl records: 20 fields: ['prompt', 'chosen', 'rejected']
Prompt-separated split: 14 3 3


In [3]:
# Local tiny GPT-2 architecture and WordLevel tokenizer: NO pretrained downloads.
from importlib import metadata
import inspect
from tokenizers import Tokenizer,models,pre_tokenizers
from transformers import PreTrainedTokenizerFast,GPT2Config,GPT2LMHeadModel,GPT2ForSequenceClassification
from datasets import Dataset
from trl import DPOTrainer,DPOConfig,GRPOTrainer,GRPOConfig,ORPOTrainer,ORPOConfig,RewardTrainer,RewardConfig,PPOTrainer,PPOConfig
VERSIONS={name:metadata.version(name) for name in ('torch','transformers','trl','datasets','peft','accelerate')}
print(VERSIONS)
assert metadata.version('trl')=='0.22.2', 'Use the pinned environment; APIs change across TRL releases.'
for cls in [DPOTrainer,GRPOTrainer,ORPOTrainer,PPOTrainer]: print(cls.__name__,inspect.signature(cls.__init__))
texts=['Question: '+r['prompt']+' Answer: '+r[s] for r in rows for s in ('chosen','rejected')]
words=sorted(set(re.findall(r'\w+|[^\w\s]', ' '.join(texts)+' What is 1 + 1 ? 2 3 4 - .')))
word_vocab={'<pad>':0,'<eos>':1,'<unk>':2,'<bos>':3,**{w:i+4 for i,w in enumerate(words)}}
backend=Tokenizer(models.WordLevel(word_vocab,unk_token='<unk>'));backend.pre_tokenizer=pre_tokenizers.Whitespace()
tokenizer=PreTrainedTokenizerFast(tokenizer_object=backend,pad_token='<pad>',eos_token='<eos>',unk_token='<unk>',bos_token='<bos>')
tokenizer.chat_template="{% for message in messages %}{{ message['role'] + ': ' + message['content'] + '\\n' }}{% endfor %}{% if add_generation_prompt %}{{ 'assistant: ' }}{% endif %}"
def lm_config():
    return GPT2Config(vocab_size=len(tokenizer),n_positions=256,n_ctx=256,n_embd=32,n_layer=1,n_head=2,pad_token_id=0,eos_token_id=1,bos_token_id=3,resid_pdrop=0.,embd_pdrop=0.,attn_pdrop=0.)
def hf_pairs(examples):
    return Dataset.from_list([{'prompt':'Question: '+r['prompt']+' Answer:','chosen':' '+r['chosen']+tokenizer.eos_token,'rejected':' '+r['rejected']+tokenizer.eos_token} for r in examples])
RUN_TRL_SMOKE = True   # two steps on randomly initialized tiny LOCAL model, CPU-friendly
RUN_PRETRAINED_GPU = False  # opt-in; downloads a real checkpoint only when True
print('Tokenizer distinct PAD/EOS:',tokenizer.pad_token_id,tokenizer.eos_token_id)
assert tokenizer.pad_token_id != tokenizer.eos_token_id

/home/akhand/Desktop/LLM FINETUNING/SLM-Lab-Small-Language-Model-Experiments/13_rlhf_ppo_dpo_orpo/RLHF_NOTEBOOKS/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


{'torch': '2.8.0+cpu', 'transformers': '4.56.2', 'trl': '0.22.2', 'datasets': '4.3.0', 'peft': '0.17.1', 'accelerate': '1.10.1'}
DPOTrainer (self, model: Union[str, torch.nn.modules.module.Module, transformers.modeling_utils.PreTrainedModel], ref_model: Union[transformers.modeling_utils.PreTrainedModel, torch.nn.modules.module.Module, str, NoneType] = None, args: Optional[trl.trainer.dpo_config.DPOConfig] = None, data_collator: Optional[transformers.data.data_collator.DataCollator] = None, train_dataset: Union[datasets.arrow_dataset.Dataset, datasets.iterable_dataset.IterableDataset, NoneType] = None, eval_dataset: Union[datasets.arrow_dataset.Dataset, datasets.iterable_dataset.IterableDataset, dict[str, Union[datasets.arrow_dataset.Dataset, datasets.iterable_dataset.IterableDataset]], NoneType] = None, processing_class: Union[transformers.tokenization_utils_base.PreTrainedTokenizerBase, transformers.image_processing_utils.BaseImageProcessor, transformers.feature_extraction_utils.Featu

In [4]:
if RUN_TRL_SMOKE:
    # First train a small scalar RM from real pairs; never use a random RM as evidence.
    tokenizer.padding_side='right'
    rm_cfg=lm_config();rm_cfg.num_labels=1
    reward_model=GPT2ForSequenceClassification(rm_cfg)
    rm_args=RewardConfig(output_dir=str(RUNTIME/'ppo_rm_cpu'),max_steps=2,per_device_train_batch_size=2,learning_rate=1e-4,
                         max_length=128,use_cpu=True,bf16=False,fp16=False,report_to='none',save_strategy='no',remove_unused_columns=False)
    rm_trainer=RewardTrainer(model=reward_model,args=rm_args,processing_class=tokenizer,train_dataset=hf_pairs(train_rows),eval_dataset=hf_pairs(valid_rows))
    rm_trainer.train();print('Reward-model held-out ranking loss:',rm_trainer.evaluate())
    reward_model.eval().requires_grad_(False)
    value_model=copy.deepcopy(reward_model).train().requires_grad_(True)
    policy=GPT2LMHeadModel(lm_config());reference=copy.deepcopy(policy).eval().requires_grad_(False)
    tokenizer.padding_side='left'
    def prompt_dataset(examples):
        return Dataset.from_list([{'input_ids':tokenizer('Question: '+r['prompt']+' Answer:',truncation=True,max_length=64)['input_ids']} for r in examples])
    ppo_args=PPOConfig(output_dir=str(RUNTIME/'ppo_cpu'),per_device_train_batch_size=2,gradient_accumulation_steps=1,
                       total_episodes=4,num_mini_batches=1,num_ppo_epochs=1,response_length=8,
                       local_rollout_forward_batch_size=2,learning_rate=1e-5,kl_coef=.05,cliprange=.2,cliprange_value=.2,
                       gamma=1.,lam=.95,vf_coef=.1,whiten_rewards=False,temperature=1.,stop_token='eos',
                       use_cpu=True,bf16=False,fp16=False,report_to='none',save_strategy='no',logging_steps=1,seed=42)
    trainer=PPOTrainer(args=ppo_args,processing_class=tokenizer,model=policy,ref_model=reference,reward_model=reward_model,
                       value_model=value_model,train_dataset=prompt_dataset(train_rows),eval_dataset=prompt_dataset(valid_rows))
    result=trainer.train();trainer.save_model(str(RUNTIME/'ppo_cpu/final'));tokenizer.save_pretrained(str(RUNTIME/'ppo_cpu/final'))
    print('Actual tiny CPU PPO run:',result)
    # PPO rollout old-logps are not the frozen-reference logps. Critic remains trainable.
    assert all(not p.requires_grad for p in reward_model.parameters())
    assert all(not p.requires_grad for p in reference.parameters())
    assert any(p.requires_grad for p in value_model.parameters())
    trained=trainer.accelerator.unwrap_model(trainer.model).policy.eval()
    query=tokenizer('Question: What is AI? Answer:',return_tensors='pt')
    with torch.no_grad():generated=trained.generate(**query,max_new_tokens=8,do_sample=False,pad_token_id=tokenizer.pad_token_id)
    print('Random-model smoke response (not meaningful):',tokenizer.decode(generated[0,query.input_ids.shape[1]:]))

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Map: 100%|██████████| 14/14 [00:00<00:00, 5601.47 examples/s]

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Map: 100%|██████████| 14/14 [00:00<00:00, 3296.30 examples/s]

Filter:   0%|          | 0/14 [00:00<?, ? examples/s]

...

/home/akhand/Desktop/LLM FINETUNING/SLM-Lab-Small-Language-Model-Experiments/13_rlhf_ppo_dpo_orpo/RLHF_NOTEBOOKS/.venv/lib/python3.13/site-packages/trl/trainer/ppo_trainer.py:517: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:306.)
  rewards[[actual_start, actual_end]] += scores


Reward-model held-out ranking loss: {'eval_loss': 0.6878741383552551, 'eval_accuracy': 0.6666666666666666, 'eval_runtime': 0.0076, 'eval_samples_per_second': 396.95, 'eval_steps_per_second': 132.317, 'epoch': 0.2857142857142857}
===training policy===


/home/akhand/Desktop/LLM FINETUNING/SLM-Lab-Small-Language-Model-Experiments/13_rlhf_ppo_dpo_orpo/RLHF_NOTEBOOKS/.venv/lib/python3.13/site-packages/trl/trainer/ppo_trainer.py:641: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  metrics["val/ratio_var"] = self.accelerator.gather_for_metrics(ratio_stats).var().item()


Step,Training Loss


Actual tiny CPU PPO run: None
Random-model smoke response (not meaningful): ignore <pad> <pad> <pad> <pad> <pad> <pad> <pad>


## Optional GPU pipeline

Disabled by default. The reward head starts new and is trained on the real comparison pairs. A tiny RM has serious generalization and length-bias limitations. Choose disjoint prompts for assessment; do not optimize against an unchecked random scorer. Periodic trainer checkpoints enable resume where supported, while the final export saves policy weights.

In [5]:
if RUN_PRETRAINED_GPU:
    assert torch.cuda.is_available()
    from transformers import AutoTokenizer,AutoModelForCausalLM,AutoModelForSequenceClassification
    MODEL_ID='HuggingFaceTB/SmolLM2-135M-Instruct'
    tok=AutoTokenizer.from_pretrained(MODEL_ID)
    # Distinct padding avoids PPO final-token pooling interpreting EOS as padding.
    if tok.pad_token_id is None or tok.pad_token_id==tok.eos_token_id:tok.add_special_tokens({'pad_token':'<|rlhf_pad|>'})
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
    rm=AutoModelForSequenceClassification.from_pretrained(MODEL_ID,num_labels=1,torch_dtype=dtype)
    rm.resize_token_embeddings(len(tok));rm.config.pad_token_id=tok.pad_token_id
    def chat_pairs(examples):
        return Dataset.from_list([{'prompt':[{'role':'user','content':r['prompt']}], 'chosen':[{'role':'assistant','content':r['chosen']}],
                                   'rejected':[{'role':'assistant','content':r['rejected']}]} for r in examples])
    tok.padding_side='right'
    rm_args=RewardConfig(output_dir=str(RUNTIME/'ppo_rm_gpu'),max_steps=5,per_device_train_batch_size=1,gradient_accumulation_steps=2,
                         learning_rate=1e-5,max_length=256,bf16=dtype==torch.bfloat16,fp16=False,report_to='none',save_strategy='no',remove_unused_columns=False)
    rm_t=RewardTrainer(model=rm,args=rm_args,processing_class=tok,train_dataset=chat_pairs(train_rows),eval_dataset=chat_pairs(valid_rows))
    rm_t.train();print(rm_t.evaluate());rm.eval().requires_grad_(False)
    # Same embedding shape and pad token in all four roles; base token resize performed
    # once before copying so added token weights match the reference exactly.
    actor=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=dtype)
    actor.resize_token_embeddings(len(tok));actor.config.pad_token_id=tok.pad_token_id;actor.config.use_cache=False
    ref=copy.deepcopy(actor).eval().requires_grad_(False)
    critic=copy.deepcopy(rm).train().requires_grad_(True)
    tok.padding_side='left';tok.truncation_side='left'
    def prompts(examples):
        return Dataset.from_list([{'input_ids':tok.apply_chat_template([{'role':'user','content':r['prompt']}],tokenize=True,add_generation_prompt=True,truncation=True,max_length=128)} for r in examples])
    args=PPOConfig(output_dir=str(RUNTIME/'ppo_gpu'),total_episodes=8,per_device_train_batch_size=2,num_mini_batches=1,num_ppo_epochs=1,
                   response_length=32,local_rollout_forward_batch_size=2,learning_rate=1e-6,kl_coef=.05,cliprange=.2,gamma=1.,lam=.95,
                   bf16=dtype==torch.bfloat16,fp16=False,report_to='none',stop_token='eos',whiten_rewards=False,save_strategy='steps',save_steps=2)
    ppo=PPOTrainer(args=args,processing_class=tok,model=actor,ref_model=ref,reward_model=rm,value_model=critic,
                   train_dataset=prompts(train_rows),eval_dataset=prompts(valid_rows))
    ppo.train();ppo.save_model(str(RUNTIME/'ppo_gpu/final'));tok.save_pretrained(str(RUNTIME/'ppo_gpu/final'))
    # Compare held-out prompts and scored KL/length/task quality with the starting actor.
    # Eight episodes + a five-step RM are workflow checks, not validated alignment.

## Exercises

1. Print response masks, old/reference logps and shaped rewards for a rollout.
2. Test RM on verbose incorrect responses.
3. Compare saved policy with reference on held-out prompts and record length/KL.

## Key takeaways

Recompute the worked examples, inspect masks and gradients, and evaluate independently of the training objective. A small training-loss decrease does not establish generalization or alignment. Record the model, data split, seeds and dependency versions before interpreting results.